# 3교시. 문서 구조 이해 및 추출 결과 정제

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/03_document_structure.ipynb)

**이번 교시 행동:** 2교시 결과를 불러와 원문은 보존하고, 공백·날짜·표 영역만 정리합니다.

**통과 증거:** `course_outputs/clean_receipt.json`

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

화면의 실행 모드를 먼저 확인합니다.

- `LIVE`: 현재 파일에 실제 모델을 실행한 결과
- `PREPARED_FALLBACK`: 공개 샘플을 사람이 검수해 둔 복구 결과
- 3분 이상 멈추면 실행을 중지하고 복구 결과로 계속합니다.
- 각 교시 끝에서 `CHECKPOINT PASS`와 산출물 파일을 확인합니다.


In [ ]:
import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
VALIDATION_MODE = os.getenv("COURSE_VALIDATE_PREPARED") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or VALIDATION_MODE:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 준비 입력을 쓰려면 "
            "USE_PREPARED_INPUT=True로 바꾸세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if VALIDATION_MODE:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())


In [ ]:
GOLDEN_OCR_TEXT = '이태리집\n거래일시 2025-10-04 12:33:37\n페퍼로니 앤 치즈 29,000 1 29,000\n토마토 파스타 14,000 1 14,000\n수제 돈가스 13,000 1 13,000\n새우 칠리치 필라 14,000 1 14,000\n콜라 2,000 3 6,000\n합계 금액 76,000\n부가세 과세물품가액 69,094\n부가세 6,906\n'
GOLDEN_RECEIPT = {'document_type': 'receipt', 'store_name': '이태리집', 'date': '2025-10-04', 'total_amount': 76000, 'items': [{'name': '페퍼로니 앤 치즈', 'quantity': 1, 'unit_price': 29000, 'line_total': 29000}, {'name': '토마토 파스타', 'quantity': 1, 'unit_price': 14000, 'line_total': 14000}, {'name': '수제 돈가스', 'quantity': 1, 'unit_price': 13000, 'line_total': 13000}, {'name': '새우 칠리치 필라', 'quantity': 1, 'unit_price': 14000, 'line_total': 14000}, {'name': '콜라', 'quantity': 3, 'unit_price': 2000, 'line_total': 6000}], 'adjustments': {'discount': 0, 'tax': 0, 'service': 0, 'rounding': 0}, 'tax_breakdown': {'mode': 'included_in_item_prices', 'supply_amount': 69094, 'vat': 6906, 'payable_total': 76000}, 'raw_values': {'store_name': '이태리집', 'date': '2025-10-04 12:33:37', 'total_amount': '76,000'}, 'cleaned_values': {'store_name': '이태리집', 'date': '2025-10-04', 'total_amount': 76000}, 'evidence': {'store_name': {'raw_value': '이태리집', 'line': 1}, 'date': {'raw_value': '거래일시 2025-10-04 12:33:37', 'line': 2}, 'total_amount': {'raw_value': '합계 금액 76,000', 'line': 8}}, 'source_mode': 'prepared_fixture_rule_extraction'}


In [ ]:
import re

previous_path = OUTPUT_DIR / "ocr_result.json"
USE_PREPARED_INPUT = VALIDATION_MODE
if not previous_path.exists() and not USE_PREPARED_INPUT:
    upload_previous_artifact("ocr_result.json")
if previous_path.exists():
    previous = json.loads(previous_path.read_text(encoding="utf-8"))
    raw_text = "\n".join(item["text"] for item in previous["items"])
    INPUT_MODE = "PREVIOUS_LESSON"
else:
    raw_text = GOLDEN_OCR_TEXT
    INPUT_MODE = "PREPARED_FALLBACK"
print("입력 모드:", INPUT_MODE)


def clean_lines(text):
    cleaned = []
    changes = []
    for raw in text.splitlines():
        normalized = re.sub(r"\s+", " ", raw.strip())
        if normalized:
            cleaned.append(normalized)
        if raw != normalized:
            changes.append({"before": raw, "after": normalized})
    groups = {"header": [], "date": [], "items": [], "total": [], "other": []}
    for line in cleaned:
        if re.search(r"\d{4}[-./]\d{1,2}[-./]\d{1,2}", line):
            groups["date"].append(line)
        elif "합계" in line:
            groups["total"].append(line)
        elif re.search(r"[\d,]+\s+\d+\s+[\d,]+$", line):
            groups["items"].append(line)
        elif not groups["header"]:
            groups["header"].append(line)
        else:
            groups["other"].append(line)
    return {
        "input_mode": INPUT_MODE,
        "raw_text": text,
        "cleaned_lines": cleaned,
        "groups": groups,
        "change_log": changes,
        "rule": "원문에 없는 값은 추가하지 않음",
    }


clean_result = clean_lines(raw_text)
output_path = OUTPUT_DIR / "clean_receipt.json"
output_path.write_text(
    json.dumps(clean_result, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
assert clean_result["raw_text"] == raw_text
print("원문 줄:", len(raw_text.splitlines()))
print("품목 후보 줄:", len(clean_result["groups"]["items"]))
print("CHECKPOINT 1/1 PASS:", output_path)
download_artifact(output_path)
